# DERMATEC — RED FINAL (Z.18 / Z.19), Kaggle — entrenamiento + reporte completo para la tesis
Receta consolidada de todo el proyecto:
- **384 px** (la mejor resolución: Z.7 fue el mejor modelo individual en Validation y Test).
- Ajuste fino continuando desde **Z7.keras** (EfficientNetV2-B0) o **Z9.keras** (MobileNetV3-L); si no están, desde Z2/Z3.
- Shades of Gray dentro del modelo, SupCon, pesos por clase, TTA, sesgo y umbral calibrados en **Validation**.
- **Aumentación de escala moderada** (zoom 1,3×–2,6×, prob. 0,5): con el acople de distancia fija la escala ya es la correcta;
  esto solo da tolerancia a pequeñas variaciones de distancia sin costo en 1× (Z.14 lo demostró).
- Reporte final: curvas de **loss y accuracy** (entrenamiento vs validación), **matrices de confusión** 4 clases y binaria,
  **F1, sensibilidad y especificidad** multiclase y binaria en **Train, Validation y Test**.
- Paquete para la App V6 (`.tflite` + `config_dermatec.json`) y `.zip` para descargar.

| Exp. | Celda 1 |
|---|---|
| **Z.18** | valores por defecto (EfficientNetV2-B0, desde z7.keras) |
| **Z.19** | `EXPERIMENTO="Z.19"`, `BACKBONE="mobilenetv3l"`, `MODELO_PARTIDA=["z9.keras","z3.keras"]` |

**Inputs de Kaggle:** DERMATEC_KAGGLE_V2 (TAR), un dataset con los `.keras` de partida (renombrados `Z7.keras`, `Z9.keras`; o `Z2.keras`, `Z3.keras`)
y (opcional) las capturas del capilógrafo. GPU T4/P100. Siempre **Run All**; al final descargar el `.zip` y **Save Version**.
El Test se evalúa una vez al final y se **reporta**; no se usa para elegir nada (sesgo y umbral salen de Validation).


# 1. CONFIGURACIÓN
Única celda que se edita. Todo lo demás lee de aquí.

In [ ]:
# ============================================================
# 1. CONFIGURACIÓN DEL EXPERIMENTO
# ============================================================
EXPERIMENTO   = "Z.18"          # Z.18 effnet 384 | Z.19 mobilenet 384 (ver introducción)
USAR_SUPCON   = True             # receta ganadora (Z.2): contrastivo
MODELO_PARTIDA = ["z7.keras", "z2.keras"]   # se usa el primero que exista en /kaggle/input (Z.19: ["z9.keras", "z3.keras"])
CONTINUAR_DESDE = "buscar"
SOLO_CACHE    = False            # True = runtime CPU: solo prepara la caché en Drive y se detiene (no gasta GPU)

# --- Red ---
BACKBONE      = "effnetv2b0"     # effnetv2b0 | effnetv2b1 | effnetv2b2 | effnetv2b3 | mobilenetv3l | resnet50
PESOS_BACKBONE = None           # al continuar no hace falta descargar ImageNet (no requiere Internet)
IMG_SIZE      = 384
USAR_SHADES_OF_GRAY = True       # Rama 2: constancia de color (Barata et al., 2015)
DROPOUT       = 0.30

# --- Entrenamiento ---
BATCH_SIZE    = 32               # 384 px en T4 (igual que Z.7 / Z.9)
EPOCAS_CABEZA = 2                # fase A: backbone congelado
EPOCAS_FT     = 12
LR_CABEZA     = 2e-3
LR_FT_MAX     = 4e-5
WEIGHT_DECAY  = 1e-2             # AdamW (decaimiento desacoplado)
LABEL_SMOOTHING = 0.05
PACIENCIA     = 8                # épocas sin mejorar Macro-F1 de Validation

# --- Aprendizaje contrastivo (solo si USAR_SUPCON) ---
SUPCON_PESO   = 0.30
SUPCON_TEMP   = 0.10
PROYECCION_DIM = 128

# --- Metas y calibración ---
META_MACRO_F1 = 0.80
META_SENS     = 0.85
SENS_CALIBRACION = 0.95          # umbral de P(maligno) se elige para sens >= este valor

# --- Tiempo de GPU (minutos) ---
PRESUPUESTO_ENTRENAMIENTO_MIN = 150

# --- Datos ---
SEED = 2026
CONTEOS_ESPERADOS = {"train": 12_994, "validation": 2_156}
TAR_BYTES_ESPERADOS = 6_748_149_760
GUARDAR_CACHE_EN_DRIVE = True    # ~450 MB; la siguiente sesión arranca en 1–2 min

CLASES = ["Nevo", "Melanoma", "BCC", "SCC"]   # indice_clase_4 = 0,1,2,3
CLASES_MALIGNAS = [1, 2, 3]
# --- Aumentación de ESCALA (capilógrafo) — serie Z.10–Z.17 ---
ZOOM_P          = 0.5            # mitad de las imágenes con zoom moderado
ZOOM_AREA       = (0.15, 0.60)   # zoom lineal ~1,3×–2,6× (con acople la escala ya es correcta)
ZOOM_JITTER     = 0.15           # desplazamiento máximo del centro del recorte (fracción del lado)
DEGRADAR_P      = 0.5            # dentro del zoom: resolución + JPEG + desenfoque + tinte
DEGRADAR_BLUR_COLOR = True       # desenfoque 3×3 (50 %) y tinte 0,9–1,1 por canal. [Verificar si estuvo activo en Z.14]
EVAL_ZOOMS      = [2.0]          # validación con zoom 2× (el criterio de época usa 1× y 2×)


# 2. ENTORNO
Librerías, GPU, precisión mixta y semillas.

In [ ]:
import os, io, gc, json, time, shutil, tarfile, hashlib, random
from pathlib import Path
from datetime import datetime
from concurrent.futures import ThreadPoolExecutor

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image
from sklearn.metrics import (f1_score, precision_recall_fscore_support, confusion_matrix,
                             balanced_accuracy_score, roc_auc_score)

import tensorflow as tf
import keras
from keras import layers

random.seed(SEED); np.random.seed(SEED); tf.random.set_seed(SEED)

GPUS = tf.config.list_physical_devices("GPU")
if not GPUS and not SOLO_CACHE:
    raise RuntimeError("❌ No hay GPU. Entorno de ejecución → Cambiar tipo → GPU, y vuelve a ejecutar.")
for g in GPUS:
    try:
        tf.config.experimental.set_memory_growth(g, True)
    except RuntimeError:
        pass   # la GPU ya estaba inicializada en esta sesión
keras.mixed_precision.set_global_policy("mixed_float16")

T0_SESION = time.time()
TAG = f"{EXPERIMENTO}_{BACKBONE}_{IMG_SIZE}px_{'SUPCON' if USAR_SUPCON else 'CE'}"
print(f"✅ TF {tf.__version__} | Keras {keras.__version__} | GPU: {GPUS[0].name if GPUS else 'NO (modo SOLO_CACHE)'}")
print(f"✅ Experimento: {TAG}")

# 3. RUTAS (Colab/Drive o Kaggle)
Detecta solo dónde está corriendo. **Colab:** busca `DERMATEC` o `DERMATEC (1)` en MyDrive y el TAR dentro o fuera de ella. **Kaggle:** busca en `/kaggle/input` los CSV y el TAR (o las imágenes ya descomprimidas) y guarda todo en `/kaggle/working`.

In [ ]:
# ============================================================
# 3. RUTAS (Colab/Drive o Kaggle) — búsqueda automática
# ============================================================
RUTA_PROYECTO_MANUAL = None   # ej: "/content/drive/MyDrive/DERMATEC"  (solo si la búsqueda falla)
RUTA_TAR_MANUAL      = None   # ej: "/content/drive/MyDrive/DATASETS/.../DERMATEC_KAGGLE_V2.tar"

# Kaggle trae instalado el paquete google.colab y Colab trae /kaggle/input: se usan marcas propias de cada entorno
EN_KAGGLE = "KAGGLE_KERNEL_RUN_TYPE" in os.environ or (
    Path("/kaggle/input").is_dir() and not Path("/var/colab/hostname").exists())
EN_COLAB = not EN_KAGGLE
TAR_PATH, IMAGES_ROOT = None, None

def buscar(raiz, condicion, prof_max=4):
    """Recorre carpetas (incluye accesos directos) hasta prof_max niveles y devuelve la primera que cumple."""
    pendientes = [(Path(raiz), 0)]
    while pendientes:
        p, d = pendientes.pop(0)
        try:
            if condicion(p): return p
            if d < prof_max:
                pendientes += [(h, d + 1) for h in sorted(p.iterdir()) if h.is_dir() and not h.name.startswith(".")]
        except (PermissionError, OSError):
            pass
    return None

if not EN_KAGGLE:
    from google.colab import drive
    drive.mount("/content/drive", force_remount=False)
    MYDRIVE = Path("/content/drive/MyDrive")
    es_trabajo = lambda p: p.name == "entreno final 19 septiembre" and (p / "00_DATASET" / "train.csv").is_file()
    tiene_tar = lambda p: (p / "DERMATEC_KAGGLE_V2.tar").is_file() and \
                          (p / "DERMATEC_KAGGLE_V2.tar").stat().st_size == TAR_BYTES_ESPERADOS

    WORK_ROOT = (Path(RUTA_PROYECTO_MANUAL) / "entreno final 19 septiembre") if RUTA_PROYECTO_MANUAL \
                else buscar(MYDRIVE, es_trabajo)
    if RUTA_TAR_MANUAL:
        TAR_PATH = Path(RUTA_TAR_MANUAL)
    else:
        sub = Path("DATASETS/05_Entrenamiento_DERMATEC/runs/KAGGLE_TRANSFER/DERMATEC_KAGGLE_V2.tar")
        rapidas = ([WORK_ROOT.parent / sub] if WORK_ROOT else []) + [MYDRIVE / sub, MYDRIVE / "DERMATEC" / sub,
                                                                      MYDRIVE / "DERMATEC (1)" / sub]
        TAR_PATH = next((t for t in rapidas if t.is_file() and t.stat().st_size == TAR_BYTES_ESPERADOS), None)
        if TAR_PATH is None:
            carpeta = buscar(MYDRIVE, tiene_tar, prof_max=7)
            TAR_PATH = carpeta / "DERMATEC_KAGGLE_V2.tar" if carpeta else None

    if WORK_ROOT is None or TAR_PATH is None:
        print("📂 Contenido de MyDrive (nivel 1 y 2):")
        for p in sorted(MYDRIVE.iterdir()):
            print("  ", p.name)
            if p.is_dir():
                try:
                    for h in sorted(p.iterdir())[:15]: print("      └─", h.name)
                except OSError: pass
        raise FileNotFoundError(
            ("❌ No encontré 'entreno final 19 septiembre/00_DATASET/train.csv'.\n" if WORK_ROOT is None else "") +
            ("❌ No encontré DERMATEC_KAGGLE_V2.tar (6 748 149 760 bytes).\n" if TAR_PATH is None else "") +
            "→ Si la carpeta está en 'Compartido conmigo': clic derecho → Organizar → Agregar acceso directo → Mi unidad.\n"
            "→ O escribe la ruta en RUTA_PROYECTO_MANUAL / RUTA_TAR_MANUAL (arriba) y vuelve a ejecutar.")
    PROJECT_ROOT = WORK_ROOT.parent
    TRAIN_CSV, VAL_CSV = WORK_ROOT / "00_DATASET" / "train.csv", WORK_ROOT / "00_DATASET" / "validation.csv"
    LOCAL = Path("/content/dermatec_z")
else:
    KIN = Path("/kaggle/input")
    WORK_ROOT = Path("/kaggle/working")
    GUARDAR_CACHE_EN_DRIVE = False
    import io as _io

    def normalizar(df):
        """Acepta el manifiesto con nombres de columna distintos y lo deja con los nombres del proyecto."""
        df = df.copy()
        low = {c.lower().strip(): c for c in df.columns}
        if "image_relpath" not in df:
            c = next((c for c in df.columns if df[c].astype(str).str.contains("images/").mean() > 0.9), None)
            if c: df["image_relpath"] = df[c]
        if "indice_clase_4" not in df:
            c = next((low[k] for k in ["label", "label_4", "clase_idx", "class_idx", "target", "y", "indice_clase"] if k in low), None)
            if c: df["indice_clase_4"] = df[c]
        if "split" not in df:
            c = next((low[k] for k in ["partition", "particion", "set", "subset", "split_final"] if k in low), None)
            if c: df["split"] = df[c]
        if not {"image_relpath", "indice_clase_4", "split"} <= set(df.columns):
            return None
        df["image_relpath"] = df["image_relpath"].astype(str).str.replace("\\", "/", regex=False)
        df["image_relpath"] = [r[r.find("images/"):] if "images/" in r else r for r in df["image_relpath"]]
        df["split"] = df["split"].astype(str).str.lower().str.strip().replace({"val": "validation", "valid": "validation"})
        if "image_id" not in df: df["image_id"] = df["image_relpath"]
        return df

    # --- imágenes: TAR o carpeta ya descomprimida ---
    TAR_PATH = next((t for t in KIN.rglob("*.tar") if t.stat().st_size == TAR_BYTES_ESPERADOS), None)

    # --- tablas candidatas: CSV sueltos en /kaggle/input y CSV dentro del TAR ---
    tablas = []
    for c in KIN.rglob("*.csv"):
        try:
            d = normalizar(pd.read_csv(c))
            if d is not None: tablas.append((str(c), d))
        except Exception:
            pass
    if TAR_PATH is not None:
        with tarfile.open(TAR_PATH, "r:") as t:
            for m in t:
                if m.isfile() and m.name.lower().endswith(".csv"):
                    try:
                        d = normalizar(pd.read_csv(_io.BytesIO(t.extractfile(m).read())))
                        if d is not None: tablas.append(("TAR:" + m.name, d))
                    except Exception:
                        pass

    por_nombre = {Path(n.split(":")[-1]).name: d for n, d in tablas}
    if "train.csv" in por_nombre and "validation.csv" in por_nombre:
        tr, va, origen = por_nombre["train.csv"], por_nombre["validation.csv"], "train.csv + validation.csv"
    else:
        cand = [(n, d) for n, d in tablas if {"train", "validation"} <= set(d["split"])]
        if not cand:
            print("CSV encontrados:", [n for n, _ in tablas] or "ninguno con columnas reconocibles")
            raise FileNotFoundError("❌ No encontré los splits ni en /kaggle/input ni dentro del TAR. Manda esta salida.")
        origen, d = max(cand, key=lambda x: len(x[1]))
        tr, va = d[d["split"] == "train"], d[d["split"] == "validation"]
    TRAIN_CSV, VAL_CSV = Path("/tmp/train.csv"), Path("/tmp/validation.csv")
    tr.to_csv(TRAIN_CSV, index=False); va.to_csv(VAL_CSV, index=False)
    print("✅ Splits desde:", origen)

    if TAR_PATH is None:
        rel0 = Path(str(tr["image_relpath"].iloc[0]))
        hallado = next(KIN.rglob(rel0.name), None)
        if hallado is None:
            raise FileNotFoundError(f"❌ No encontré el TAR ni la imagen {rel0} en /kaggle/input.")
        IMAGES_ROOT = hallado.parents[len(rel0.parts) - 1]
    LOCAL = Path("/tmp/dermatec_z")

LOCAL.mkdir(parents=True, exist_ok=True)
EXP_Z_ROOT = WORK_ROOT / "EXPERIMENTOS_Z"
RUN_DIR = EXP_Z_ROOT / TAG / datetime.now().strftime("%Y%m%d_%H%M%S")
if not SOLO_CACHE:
    RUN_DIR.mkdir(parents=True, exist_ok=True)
CACHE_DRIVE = WORK_ROOT / "CACHE" / f"CACHE_RGB_{IMG_SIZE}px_v1.tar"

print("✅ Entorno  :", "Kaggle" if EN_KAGGLE else "Colab + Drive")
print("✅ Trabajo  :", WORK_ROOT)
print("✅ Imágenes :", TAR_PATH or IMAGES_ROOT)
print("✅ Salida   :", RUN_DIR)


In [ ]:
# ---- Resolver el modelo de partida: el primero de MODELO_PARTIDA que exista ----
raices = [Path("/kaggle/input")] if EN_KAGGLE else [WORK_ROOT]
todos = [k for r in raices if r.is_dir() for k in r.rglob("*.keras")]
CONTINUAR_DESDE = next((str(k) for nombre in MODELO_PARTIDA for k in todos if k.name.lower() == nombre.lower()), None)
if CONTINUAR_DESDE is None:
    raise FileNotFoundError(f"❌ No encontré {MODELO_PARTIDA}. .keras disponibles: {[str(k) for k in todos]}")
print("✅ Modelo de partida:", CONTINUAR_DESDE)


# 4. DATASET
Solo Train y Validation. Verifica conteos, etiquetas y que no haya fugas entre particiones.

In [ ]:
train_df = pd.read_csv(TRAIN_CSV)
val_df   = pd.read_csv(VAL_CSV)

for nombre, df in [("train", train_df), ("validation", val_df)]:
    assert len(df) == CONTEOS_ESPERADOS[nombre], f"❌ {nombre}: {len(df)} filas"
    assert set(df["indice_clase_4"].astype(int)) == {0, 1, 2, 3}, f"❌ etiquetas {nombre}"
    assert set(df["split"].astype(str).str.lower()) == {nombre}, f"❌ split {nombre}"

for col in [c for c in ["image_relpath", "sha256", "group_id_split"] if c in train_df and c in val_df]:
    fuga = set(train_df[col].astype(str)) & set(val_df[col].astype(str))
    assert not fuga, f"❌ Fuga Train/Validation en {col}: {len(fuga)}"

Y_TRAIN = train_df["indice_clase_4"].astype(int).values
Y_VAL   = val_df["indice_clase_4"].astype(int).values

# Pesos por clase sqrt-inverse (misma fórmula de Y.1 para comparar)
n_c = np.bincount(Y_TRAIN, minlength=4)
PESOS_CLASE = np.sqrt(len(Y_TRAIN) / (4 * n_c)).astype("float32")

print(pd.DataFrame({"Train": n_c, "Validation": np.bincount(Y_VAL, minlength=4),
                    "peso": PESOS_CLASE.round(3)}, index=CLASES))
print("✅ Sin fugas. 🔒 Test no leído.")

# 5. CACHÉ DE IMÁGENES
Decodifica y redimensiona **una sola vez** Train+Validation a `IMG_SIZE` (JPEG q95 en disco local).
Esta era la causa de los 4 s/paso: cada época se decodificaban ~15 000 imágenes a resolución completa con 2 CPU.
Si la caché ya existe en Drive, solo se copia (1–2 min). Las imágenes de Test se saltan sin decodificarse.

In [ ]:
CACHE_DIR = LOCAL / f"cache_{IMG_SIZE}"
necesarias = set(train_df["image_relpath"].astype(str)) | set(val_df["image_relpath"].astype(str))
faltan = lambda: {r for r in necesarias if not (CACHE_DIR / r).is_file()}

def adecuar_imagen(datos: bytes, size: int = IMG_SIZE) -> Image.Image:
    """Misma función que usará la Raspberry: RGB → resize cuadrado (bilineal con antialias de PIL)."""
    im = Image.open(io.BytesIO(datos))
    if im.format == "JPEG":
        im.draft("RGB", (size * 2, size * 2))      # decodificación JPEG rápida a escala reducida
    return im.convert("RGB").resize((size, size), Image.BILINEAR)

t0 = time.time()
if faltan() and CACHE_DRIVE.is_file():
    print("📦 Copiando caché desde Drive...")
    with tarfile.open(CACHE_DRIVE, "r:") as t:
        try:
            t.extractall(CACHE_DIR, filter="data")
        except TypeError:
            t.extractall(CACHE_DIR)

pendientes = faltan()
if pendientes:
    print(f"🛠  Construyendo caché: {len(pendientes):,} imágenes (solo Train+Validation)...")
    def guardar(rel, datos):
        dst = CACHE_DIR / rel
        dst.parent.mkdir(parents=True, exist_ok=True)
        adecuar_imagen(datos).save(dst.with_suffix(".tmp.jpg"), "JPEG", quality=95)
        os.replace(dst.with_suffix(".tmp.jpg"), dst)
    hechas, futuros = 0, []
    if TAR_PATH is None:                                   # Kaggle con imágenes descomprimidas
        with ThreadPoolExecutor(max_workers=max(4, os.cpu_count() * 2)) as pool:
            list(pool.map(lambda r: guardar(r, (IMAGES_ROOT / r).read_bytes()), sorted(pendientes)))
    else:
      with tarfile.open(TAR_PATH, "r:") as tar, ThreadPoolExecutor(max_workers=max(4, os.cpu_count() * 2)) as pool:
          for m in tar:
              if not m.isfile():
                  continue
              nombre = m.name.replace("\\", "/")
              i = nombre.find("images/")
              rel = nombre[i:] if i >= 0 else None
              if rel not in pendientes:
                  continue                                  # Test y otros archivos: ni se leen como imagen
              futuros.append(pool.submit(guardar, rel, tar.extractfile(m).read()))
              if len(futuros) >= 256:
                  for f in futuros: f.result()
                  hechas += len(futuros); futuros = []
                  if hechas % 2048 < 256:
                      print(f"   {hechas:,}/{len(pendientes):,}  ({time.time()-t0:.0f}s)")
          for f in futuros: f.result()

    if GUARDAR_CACHE_EN_DRIVE and not CACHE_DRIVE.is_file():
        CACHE_DRIVE.parent.mkdir(parents=True, exist_ok=True)
        tmp = CACHE_DRIVE.with_suffix(".tmp")
        with tarfile.open(tmp, "w") as t:
            t.add(CACHE_DIR / "images", arcname="images")
        os.replace(tmp, CACHE_DRIVE)
        print("💾 Caché guardada en Drive:", CACHE_DRIVE)

assert not faltan(), f"❌ Faltan {len(faltan())} imágenes en caché"
train_df["local"] = [str(CACHE_DIR / r) for r in train_df["image_relpath"].astype(str)]
val_df["local"]   = [str(CACHE_DIR / r) for r in val_df["image_relpath"].astype(str)]
print(f"✅ Caché lista: {len(necesarias):,} imágenes {IMG_SIZE}×{IMG_SIZE} en {time.time()-t0:.0f}s")
if SOLO_CACHE:
    class CacheListaDetenerAqui(Exception): pass
    raise CacheListaDetenerAqui(f"✅ Caché guardada en Drive ({CACHE_DRIVE.name}). "
                                "Esto NO es un error: cambia a GPU, pon SOLO_CACHE = False y ejecuta todo.")

# 6. RAMA 2 — ADECUACIÓN DE ENTRADA (procesamiento suave)
**Shades of Gray** (Minkowski p = 6): estima el color del iluminante y lo neutraliza. Es la normalización usada por los mejores sistemas ISIC (Barata et al., 2015; Gessert et al., 2020) y ataca directamente el *domain shift* del capilógrafo.
Va **dentro del modelo**, así que la Raspberry solo hace `resize` y le pasa la imagen: entrenamiento y despliegue usan exactamente la misma operación.

Lo que **no** entra a la red (decidido con X.1/X.2 y el estado del arte): máscaras, ROI, radiomics, inpainting, CLAHE fuerte. Eso sigue siendo trabajo de la Rama 1.

In [ ]:
@keras.saving.register_keras_serializable(package="dermatec")
class ShadesOfGray(layers.Layer):
    """Constancia de color Shades of Gray. Entrada/salida en [0,255]. Conserva el brillo medio."""
    def __init__(self, p=6, **kw):
        kw.setdefault("dtype", "float32")
        super().__init__(**kw); self.p = p
    def call(self, x):
        x = keras.ops.cast(x, "float32")
        xn = keras.ops.clip(x / 255.0, 1e-6, 1.0)
        ilum = keras.ops.power(keras.ops.mean(keras.ops.power(xn, self.p), axis=(1, 2), keepdims=True), 1.0 / self.p)
        ganancia = keras.ops.mean(ilum, axis=-1, keepdims=True) / (ilum + 1e-6)
        return keras.ops.clip(x * ganancia, 0.0, 255.0)
    def get_config(self):
        return {**super().get_config(), "p": self.p}

@keras.saving.register_keras_serializable(package="dermatec")
class CaffePreprocess(layers.Layer):
    """Preprocesado de ResNet50 (RGB→BGR y resta de media ImageNet)."""
    def call(self, x):
        x = keras.ops.cast(x, self.compute_dtype)[..., ::-1]
        return x - keras.ops.convert_to_tensor([103.939, 116.779, 123.68], dtype=self.compute_dtype)

# Vista rápida: 2 imágenes por clase, original vs Shades of Gray
sog = ShadesOfGray()
fig, ax = plt.subplots(2, 8, figsize=(20, 5.5))
muestra = val_df.groupby("indice_clase_4").sample(2, random_state=SEED)
for j, (_, fila) in enumerate(muestra.iterrows()):
    img = np.asarray(Image.open(fila["local"]), dtype="float32")[None]
    ax[0, j].imshow(img[0].astype("uint8")); ax[0, j].set_title(CLASES[fila["indice_clase_4"]]); ax[0, j].axis("off")
    ax[1, j].imshow(np.asarray(sog(img))[0].astype("uint8")); ax[1, j].set_title("Shades of Gray"); ax[1, j].axis("off")
plt.tight_layout(); plt.savefig(RUN_DIR / "RAMA2_muestra.png", dpi=90); plt.show()

# 7. PIPELINE tf.data
Lee la caché local y aplica la aumentación en CPU con operaciones exactas y baratas: 8 simetrías (rot90 + espejo), recorte aleatorio 80–100 % y brillo/contraste/saturación suaves. Sustituye las capas de rotación/zoom/traslación con interpolación de Keras, que eran el cuello de botella (~6 min/época en Z.1).

In [ ]:
AUTOTUNE = tf.data.AUTOTUNE

def leer(ruta):
    img = tf.io.decode_jpeg(tf.io.read_file(ruta), channels=3)
    img.set_shape([IMG_SIZE, IMG_SIZE, 3])
    return tf.cast(img, tf.float32)

def zoom_capilografo(img):
    """Recorte cercano al centro (zoom 1/√área: ~1,5×–4× con ZOOM_AREA (0,06; 0,45)) + degradación opcional tipo capilógrafo."""
    area = tf.random.uniform([], ZOOM_AREA[0], ZOOM_AREA[1])
    lado = tf.cast(tf.sqrt(area) * IMG_SIZE, tf.int32)
    libre = IMG_SIZE - lado
    j = tf.cast(ZOOM_JITTER * IMG_SIZE, tf.int32)
    c = libre // 2
    oy = tf.clip_by_value(c + tf.random.uniform([], -j, j + 1, tf.int32), 0, libre)
    ox = tf.clip_by_value(c + tf.random.uniform([], -j, j + 1, tf.int32), 0, libre)
    img = tf.image.resize(tf.image.crop_to_bounding_box(img, oy, ox, lado, lado), [IMG_SIZE, IMG_SIZE])
    def degradar(x):
        f = tf.random.uniform([], 0.4, 0.7)
        pequeno = tf.cast(f * IMG_SIZE, tf.int32)
        x = tf.image.resize(tf.image.resize(x, [pequeno, pequeno], antialias=True), [IMG_SIZE, IMG_SIZE])
        x = tf.image.random_jpeg_quality(tf.clip_by_value(x / 255.0, 0.0, 1.0), 55, 90) * 255.0
        if DEGRADAR_BLUR_COLOR:
            x = tf.cond(tf.random.uniform([]) < 0.5,
                        lambda: tf.nn.avg_pool2d(x[None], 3, 1, "SAME")[0], lambda: x)   # desenfoque 3×3
            x = tf.clip_by_value(x * tf.random.uniform([1, 1, 3], 0.9, 1.1), 0.0, 255.0)  # tinte por canal
        return x
    return tf.cond(tf.random.uniform([]) < DEGRADAR_P, lambda: degradar(img), lambda: img)

def recorte_suave(img):
    lado = tf.cast(tf.random.uniform([], 0.80, 1.0) * IMG_SIZE, tf.int32)
    return tf.image.resize(tf.image.random_crop(img, [lado, lado, 3]), [IMG_SIZE, IMG_SIZE])

def aumentar(img):
    img = tf.image.rot90(img, tf.random.uniform([], 0, 4, tf.int32))
    img = tf.image.random_flip_left_right(img)
    img = tf.cond(tf.random.uniform([]) < ZOOM_P, lambda: zoom_capilografo(img), lambda: recorte_suave(img))
    img.set_shape([IMG_SIZE, IMG_SIZE, 3])
    img = img / 255.0
    img = tf.image.random_brightness(img, 0.10)
    img = tf.image.random_contrast(img, 0.85, 1.15)
    img = tf.image.random_saturation(tf.clip_by_value(img, 0.0, 1.0), 0.8, 1.2)
    return tf.clip_by_value(img, 0.0, 1.0) * 255.0

def dataset(df, entrenamiento):
    rutas = df["local"].values
    y = df["indice_clase_4"].astype("int32").values
    ds = tf.data.Dataset.from_tensor_slices((rutas, y))
    if entrenamiento:
        ds = ds.shuffle(len(df), seed=SEED, reshuffle_each_iteration=True)
    def ejemplo(r, c):
        oh = tf.one_hot(c, 4)
        w = tf.gather(PESOS_CLASE, c) if entrenamiento else tf.constant(1.0)
        ys, ws = {"cls": oh}, {"cls": w}
        if USAR_SUPCON:
            ys["proj"], ws["proj"] = oh, tf.constant(1.0)
        img = aumentar(leer(r)) if entrenamiento else leer(r)
        return img, ys, ws
    ds = ds.map(ejemplo, num_parallel_calls=AUTOTUNE)
    return ds.batch(BATCH_SIZE, drop_remainder=entrenamiento).prefetch(AUTOTUNE)

train_ds = dataset(train_df, True)
val_img_ds = (tf.data.Dataset.from_tensor_slices(val_df["local"].values)
              .map(leer, num_parallel_calls=AUTOTUNE).batch(64).prefetch(AUTOTUNE))

xb, yb, wb = next(iter(train_ds))
print("✅ Batch:", xb.shape, "| pixeles", float(tf.reduce_min(xb)), "-", float(tf.reduce_max(xb)),
      "| pesos", np.unique(np.round(wb["cls"].numpy(), 3)))

# 8. MODELO
- **Despliegue** (lo que va a la Raspberry): `imagen → Shades of Gray → backbone → GAP → Dropout → Dense(4) softmax`.
- **Entrenamiento**: mismo modelo (la aumentación ya viene del pipeline) y, en Z.2, una cabeza de proyección contrastiva que **se descarta** al exportar (no pesa en la Pi).

In [ ]:
def crear_backbone(nombre, s):
    kw = dict(include_top=False, weights=PESOS_BACKBONE, input_shape=(s, s, 3), pooling="avg")
    if nombre == "effnetv2b0":   return keras.applications.EfficientNetV2B0(include_preprocessing=True, **kw), None
    if nombre == "effnetv2b1":   return keras.applications.EfficientNetV2B1(include_preprocessing=True, **kw), None
    if nombre == "effnetv2b2":   return keras.applications.EfficientNetV2B2(include_preprocessing=True, **kw), None
    if nombre == "effnetv2b3":   return keras.applications.EfficientNetV2B3(include_preprocessing=True, **kw), None
    if nombre == "mobilenetv3l": return keras.applications.MobileNetV3Large(include_preprocessing=True, **kw), None
    if nombre == "resnet50":     return keras.applications.ResNet50(**kw), CaffePreprocess()
    raise ValueError(nombre)

def construir_modelos():
    backbone, prep = crear_backbone(BACKBONE, IMG_SIZE)
    entrada = keras.Input((IMG_SIZE, IMG_SIZE, 3), name="imagen_rgb_0_255")
    x = ShadesOfGray(name="rama2_shades_of_gray")(entrada) if USAR_SHADES_OF_GRAY else entrada
    if prep is not None: x = prep(x)
    extractor = keras.Model(entrada, backbone(x), name="extractor")

    drop = layers.Dropout(DROPOUT, name="dropout")
    densa = layers.Dense(4, name="logits")
    softmax = layers.Activation("softmax", dtype="float32", name="cls")

    despliegue = keras.Model(entrada, softmax(densa(drop(extractor(entrada)))), name="dermatec_clasificador")

    ent = keras.Input((IMG_SIZE, IMG_SIZE, 3))
    f = extractor(ent)
    salidas = {"cls": softmax(densa(drop(f)))}
    if USAR_SUPCON:
        z = layers.Dense(256, activation="relu", name="proj_1")(f)
        salidas["proj"] = layers.Dense(PROYECCION_DIM, dtype="float32", name="proj")(z)
    entreno = keras.Model(ent, salidas, name="dermatec_entreno")
    return entreno, despliegue, backbone

def supcon_loss(y_true, z):
    """Supervised Contrastive Loss (Khosla et al., 2020) sobre las 4 clases. Devuelve pérdida por muestra."""
    z = tf.math.l2_normalize(tf.cast(z, tf.float32), axis=1)
    y = tf.argmax(y_true, axis=1)
    n = tf.shape(z)[0]
    yo = tf.eye(n)
    logits = tf.matmul(z, z, transpose_b=True) / SUPCON_TEMP - 1e9 * yo
    log_prob = logits - tf.reduce_logsumexp(logits, axis=1, keepdims=True)
    pos = tf.cast(tf.equal(y[:, None], y[None, :]), tf.float32) - yo
    n_pos = tf.reduce_sum(pos, axis=1)
    valido = tf.cast(n_pos > 0, tf.float32)
    media_pos = tf.reduce_sum(pos * log_prob, axis=1) / tf.maximum(n_pos, 1.0)
    escala = tf.cast(n, tf.float32) / tf.maximum(tf.reduce_sum(valido), 1.0)
    return -media_pos * valido * escala

modelo, modelo_despliegue, backbone = construir_modelos()
print(f"✅ Parámetros modelo de despliegue: {modelo_despliegue.count_params():,}")
print(f"✅ Salidas de entrenamiento: {list(modelo.output.keys()) if isinstance(modelo.output, dict) else 'cls'}")

# 9. ENTRENAMIENTO
- **Continuación** (`CONTINUAR_DESDE`): carga los pesos del mejor modelo de un experimento anterior, salta la Fase A y hace solo el tramo final de LR bajo. El punto de partida se evalúa primero y cuenta como "mejor", así que la continuación **nunca puede devolver algo peor** que lo que ya tenías.
- **Fase A** (2 épocas, solo desde cero): backbone congelado, entrena solo la cabeza.
- **Fase B**: se descongela **todo** el backbone (las BatchNorm siguen fijas), AdamW con *warm-up* de 1 época y decaimiento coseno.
- Cada época se mide el **Macro-F1 real sobre las 2 156 imágenes de Validation** (no el promedio por lotes), y con él se guarda el mejor modelo y se decide el *early stopping*.
- Si se acerca el límite de GPU, el entrenamiento se corta solo para que siempre alcancen a correr la evaluación y el guardado.

In [ ]:
# ---- Validation con zoom 2× para el criterio de selección de época ----
def recorte_central(factor):
    def f(x):
        lado = int(round(IMG_SIZE / factor)); o = (IMG_SIZE - lado) // 2
        return tf.image.resize(x[:, o:o + lado, o:o + lado, :], [IMG_SIZE, IMG_SIZE])
    return f

VAL_ZOOM2 = val_img_ds.map(recorte_central(2.0))
print("✅ VAL_ZOOM2 listo (criterio de época = media de Macro-F1 a 1× y 2×)")


In [ ]:
def _id(x):   return x
def _lr(x):   return tf.image.flip_left_right(x)
def _ud(x):   return tf.image.flip_up_down(x)
def _lrud(x): return tf.image.flip_up_down(tf.image.flip_left_right(x))
VISTAS_TTA = [_id, _lr, _ud, _lrud]

def predecir(m, ds=val_img_ds, tta=False):
    if not tta:
        return m.predict(ds, verbose=0)
    return np.mean([m.predict(ds.map(v), verbose=0) for v in VISTAS_TTA], axis=0)

class MonitorValidation(keras.callbacks.Callback):
    def __init__(self):
        super().__init__()
        self.mejor, self.sin_mejora, self.historia, self.fase = -1.0, 0, [], "A"
        self.ruta_mejor = LOCAL / f"{TAG}_mejor.weights.h5"
    def on_epoch_begin(self, epoch, logs=None):
        self.t_ep = time.time()
    def on_epoch_end(self, epoch, logs=None):
        p = predecir(modelo_despliegue)
        pred = p.argmax(1)
        # Criterio de selección: promedio del Macro-F1 a 1× y a 2× (recorte central), para premiar la robustez al zoom
        mf1 = 0.5 * (f1_score(Y_VAL, pred, average="macro")
                     + f1_score(Y_VAL, modelo_despliegue.predict(VAL_ZOOM2, verbose=0).argmax(1), average="macro"))
        f1c = f1_score(Y_VAL, pred, average=None)
        sens = ((1 - p[:, 0] >= 0.5) & (Y_VAL > 0)).sum() / (Y_VAL > 0).sum()
        minutos = (time.time() - T0_ENTRENO) / 60
        val_loss = float(-np.mean(np.log(p[np.arange(len(Y_VAL)), Y_VAL] + 1e-9)))
        val_acc = float((pred == Y_VAL).mean())
        fila = {"fase": self.fase, "epoca": len(self.historia) + 1, "val_macro_f1": mf1, "val_sens@0.5": sens,
                "train_loss_cls": logs.get("cls_loss", logs.get("loss")), "train_acc": logs.get("cls_accuracy", logs.get("accuracy")),
                "val_loss": val_loss, "val_acc": val_acc,
                **{f"f1_{c}": v for c, v in zip(CLASES, f1c)},
                "train_loss": logs.get("loss"), "lr": float(keras.ops.convert_to_numpy(self.model.optimizer.learning_rate)),
                "seg_epoca": time.time() - self.t_ep, "min_total": minutos}
        self.historia.append(fila)
        pd.DataFrame(self.historia).to_csv(RUN_DIR / "historia_entrenamiento.csv", index=False)
        marca = ""
        if mf1 > self.mejor:
            self.mejor, self.sin_mejora, marca = mf1, 0, "  ⭐ mejor"
            modelo_despliegue.save_weights(self.ruta_mejor)
        else:
            self.sin_mejora += 1
        print(f"  [{self.fase}] ép {fila['epoca']:2d} | Macro-F1 {mf1:.4f} | sens {sens:.3f} | "
              + " ".join(f"{c[:3]} {v:.2f}" for c, v in zip(CLASES, f1c))
              + f" | {fila['seg_epoca']:.0f}s | {minutos:.0f} min{marca}")
        if self.fase == "B" and self.sin_mejora >= PACIENCIA:
            print("  ⏹ Early stopping."); self.model.stop_training = True
        if minutos > PRESUPUESTO_ENTRENAMIENTO_MIN:
            print("  ⏱ Presupuesto de GPU alcanzado: se detiene para evaluar y guardar."); self.model.stop_training = True

def compilar(lr):
    perdidas = {"cls": keras.losses.CategoricalCrossentropy(label_smoothing=LABEL_SMOOTHING)}
    pesos = {"cls": 1.0}
    if USAR_SUPCON:
        perdidas["proj"], pesos["proj"] = supcon_loss, SUPCON_PESO
    modelo.compile(optimizer=keras.optimizers.AdamW(learning_rate=lr, weight_decay=WEIGHT_DECAY),
                   loss=perdidas, loss_weights=pesos, metrics={"cls": ["accuracy"]})

monitor = MonitorValidation()
T0_ENTRENO = time.time()

if CONTINUAR_DESDE:
    # ---------------- PUNTO DE PARTIDA ----------------
    ref = Path(str(CONTINUAR_DESDE))
    if not (ref.suffix == ".keras" and ref.is_file()):
        previos = sorted((EXP_Z_ROOT / str(CONTINUAR_DESDE)).glob("*/dermatec_clasificador.keras"))
        if not previos:
            raise FileNotFoundError(f"❌ No hay dermatec_clasificador.keras en {EXP_Z_ROOT / str(CONTINUAR_DESDE)}")
        ref = previos[-1]
    modelo_despliegue.set_weights(keras.saving.load_model(ref).get_weights())
    p0 = predecir(modelo_despliegue)
    monitor.mejor = f1_score(Y_VAL, p0.argmax(1), average="macro")
    modelo_despliegue.save_weights(monitor.ruta_mejor)
    print(f"✅ Continuación desde: {ref}\n   Macro-F1 de partida (argmax): {monitor.mejor:.4f}")
    monitor.mejor = -1.0   # corrección Z.12: sin esto el monitor nunca guardaba una época nueva (Z.10/Z.11 inválidos)
else:
    # ---------------- FASE A ----------------
    print("=" * 100, "\nFASE A — backbone congelado\n" + "=" * 100)
    backbone.trainable = False
    compilar(LR_CABEZA)
    modelo.fit(train_ds, epochs=EPOCAS_CABEZA, callbacks=[monitor], verbose=0)

# ---------------- FASE B ----------------
print("=" * 100, "\nFASE B — fine-tuning completo\n" + "=" * 100)
backbone.trainable = True
for capa in backbone.layers:
    if isinstance(capa, layers.BatchNormalization):
        capa.trainable = False
pasos = len(train_df) // BATCH_SIZE
lr_sched = keras.optimizers.schedules.CosineDecay(
    initial_learning_rate=LR_FT_MAX / 20, decay_steps=pasos * (EPOCAS_FT - 1), alpha=0.01,
    warmup_target=LR_FT_MAX, warmup_steps=pasos // 4 if CONTINUAR_DESDE else pasos)
monitor.fase = "B"
compilar(lr_sched)
modelo.fit(train_ds, epochs=EPOCAS_FT, callbacks=[monitor], verbose=0)

modelo_despliegue.load_weights(monitor.ruta_mejor)
print(f"\n✅ Entrenamiento terminado en {(time.time()-T0_ENTRENO)/60:.1f} min | mejor Macro-F1 Validation: {monitor.mejor:.4f}")

# 10. EVALUACIÓN EN VALIDATION
Se reportan cuatro lecturas del mismo modelo:
1. **argmax** directo.
2. **TTA** (promedio de 4 volteos; en la Pi son 4 inferencias).
3. **Sesgo por clase calibrado**: 3 números que ajustan la prioridad de cada clase para maximizar Macro-F1 (se optimiza sobre Validation → Test lo certificará).
4. **Regla jerárquica coherente**: P(maligno) = P(Mel)+P(BCC)+P(SCC) con umbral para sensibilidad ≥ `SENS_CALIBRACION`. Si la clase más probable es Nevo pero P(maligno) supera el umbral, el caso se marca **"Nevo — sospechoso, revisar"** y cuenta como maligno. Así el binario y el multiclase nunca se contradicen.

In [ ]:
def metricas(p, sesgo=np.zeros(4), umbral=0.5):
    pred = (np.log(p + 1e-9) + sesgo).argmax(1)
    pm = 1 - p[:, 0]
    y_bin, pred_bin = (Y_VAL > 0).astype(int), (pm >= umbral).astype(int)
    tp = ((pred_bin == 1) & (y_bin == 1)).sum(); tn = ((pred_bin == 0) & (y_bin == 0)).sum()
    P, R, F, _ = precision_recall_fscore_support(Y_VAL, pred, labels=[0, 1, 2, 3], zero_division=0)
    return {"macro_f1": f1_score(Y_VAL, pred, average="macro"), "balanced_acc": balanced_accuracy_score(Y_VAL, pred),
            "accuracy": (pred == Y_VAL).mean(), "sens": tp / y_bin.sum(), "espec": tn / (1 - y_bin).sum(),
            "f1_bin": f1_score(y_bin, pred_bin), "auc_bin": roc_auc_score(y_bin, pm), "umbral": umbral,
            "por_clase": {c: {"P": P[i], "R": R[i], "F1": F[i]} for i, c in enumerate(CLASES)},
            "pred": pred, "pred_bin": pred_bin}

def umbral_para_sens(p, sens_min):
    pm, y_bin = 1 - p[:, 0], Y_VAL > 0
    for t in np.arange(0.95, 0.04, -0.005):
        if ((pm >= t) & y_bin).sum() / y_bin.sum() >= sens_min:
            return float(t)
    return 0.05

def calibrar_sesgo(p, pasadas=3):
    b, lp = np.zeros(4), np.log(p + 1e-9)
    for _ in range(pasadas):
        for c in [1, 2, 3]:
            mejores = [(f1_score(Y_VAL, (lp + np.where(np.arange(4) == c, v, b)).argmax(1), average="macro"), v)
                       for v in np.arange(-1.5, 1.51, 0.05)]
            b[c] = max(mejores)[1]
    return b

P_PLANO = predecir(modelo_despliegue)
P_TTA   = predecir(modelo_despliegue, tta=True)
lecturas = {}
for nombre, p in [("argmax", P_PLANO), ("TTA", P_TTA)]:
    lecturas[nombre] = metricas(p, umbral=umbral_para_sens(p, SENS_CALIBRACION))
P_FINAL = P_TTA if lecturas["TTA"]["macro_f1"] >= lecturas["argmax"]["macro_f1"] else P_PLANO
USA_TTA = P_FINAL is P_TTA
SESGO = calibrar_sesgo(P_FINAL)
UMBRAL = umbral_para_sens(P_FINAL, SENS_CALIBRACION)
lecturas["TTA+sesgo" if USA_TTA else "argmax+sesgo"] = metricas(P_FINAL, SESGO, UMBRAL)
FINAL = list(lecturas.values())[-1]

print(f"{'Lectura':15s} {'Macro-F1':>9s} {'BalAcc':>7s} {'Sens':>6s} {'Espec':>6s} {'F1bin':>6s} {'AUC':>6s}  F1: " + " ".join(f"{c:>8s}" for c in CLASES))
for k, m in lecturas.items():
    print(f"{k:15s} {m['macro_f1']:9.4f} {m['balanced_acc']:7.4f} {m['sens']:6.3f} {m['espec']:6.3f} {m['f1_bin']:6.3f} {m['auc_bin']:6.3f}      "
          + " ".join(f"{m['por_clase'][c]['F1']:8.3f}" for c in CLASES))
print(f"\nSesgo calibrado (Nevo, Mel, BCC, SCC): {np.round(SESGO, 2)} | umbral P(maligno): {UMBRAL:.3f}")

# ---- Regla jerárquica coherente ----
sospechosos = (FINAL["pred"] == 0) & (FINAL["pred_bin"] == 1)
print(f"Casos 'Nevo — sospechoso, revisar': {sospechosos.sum()} "
      f"(de ellos realmente malignos: {(Y_VAL[sospechosos] > 0).sum()})")

# ---- Matrices de confusión ----
fig, ax = plt.subplots(1, 2, figsize=(13, 5))
cm = confusion_matrix(Y_VAL, FINAL["pred"])
for a, M, tit, fmt in [(ax[0], cm, "Conteos", "d"), (ax[1], cm / cm.sum(1, keepdims=True), "Normalizada por fila", ".2f")]:
    a.imshow(M, cmap="Blues")
    for i in range(4):
        for j in range(4):
            a.text(j, i, format(M[i, j], fmt), ha="center", va="center", color="white" if M[i, j] > M.max() / 2 else "black")
    a.set_xticks(range(4), CLASES); a.set_yticks(range(4), CLASES); a.set_xlabel("Predicción"); a.set_ylabel("Real"); a.set_title(tit)
plt.suptitle(f"{TAG} — Validation — Macro-F1 {FINAL['macro_f1']:.4f}")
plt.tight_layout(); plt.savefig(RUN_DIR / "matriz_confusion.png", dpi=110); plt.show()

h = pd.DataFrame(monitor.historia)
plt.figure(figsize=(8, 4)); plt.plot(h["epoca"], h["val_macro_f1"], "o-", label="Macro-F1 Validation")
plt.plot(h["epoca"], h["val_sens@0.5"], "s--", label="Sensibilidad @0.5")
plt.axhline(META_MACRO_F1, color="r", ls=":", label="meta 0.80"); plt.legend(); plt.xlabel("época"); plt.grid(alpha=.3)
plt.savefig(RUN_DIR / "curva_validation.png", dpi=110); plt.show()

In [ ]:
# ============================================================
# 10b. VALIDACIÓN CON ZOOM + CAPTURAS DEL CAPILÓGRAFO (modelo nuevo vs modelo de partida)
# ============================================================
def recorte_central(factor):
    def f(x):
        lado = int(round(IMG_SIZE / factor)); o = (IMG_SIZE - lado) // 2
        return tf.image.resize(x[:, o:o + lado, o:o + lado, :], [IMG_SIZE, IMG_SIZE])
    return f

base = keras.saving.load_model(CONTINUAR_DESDE, compile=False)
filas = []
for nombre, m in [("partida", base), ("nuevo", modelo_despliegue)]:
    for z in [1.0] + list(EVAL_ZOOMS):
        ds = val_img_ds if z == 1.0 else val_img_ds.map(recorte_central(z))
        p = np.mean([m.predict(ds.map(v), verbose=0) for v in VISTAS_TTA], axis=0)
        s_, u_ = calibrar_sesgo(p), umbral_para_sens(p, SENS_CALIBRACION)
        mm = metricas(p, s_, u_)
        # también con el sesgo/umbral de zoom 1× (lo que pasaría en el dispositivo)
        filas.append({"modelo": nombre, "zoom": z, "macro_f1": mm["macro_f1"], "sens": mm["sens"], "espec": mm["espec"],
                      "f1_nevo": mm["por_clase"]["Nevo"]["F1"], "nevos_como_malignos_%": 100 * ((1 - p[Y_VAL == 0, 0]) >= 0.495).mean()})
tabla_zoom = pd.DataFrame(filas)
print(tabla_zoom.round(3).to_string(index=False))
tabla_zoom.to_csv(RUN_DIR / "validacion_zoom.csv", index=False)

# ---- Capturas propias del capilógrafo (sube la carpeta como dataset de Kaggle; nombre con "captura") ----
caps = [p for r in [Path("/kaggle/input")] if r.is_dir() for p in r.rglob("*")
        if "captura" in str(p).lower() and p.suffix.lower() in {".jpg", ".jpeg", ".png"}]
if caps:
    def cargar(ruta):
        im = Image.open(ruta)
        if im.format == "JPEG": im.draft("RGB", (IMG_SIZE * 2, IMG_SIZE * 2))
        return np.asarray(im.convert("RGB").resize((IMG_SIZE, IMG_SIZE), Image.BILINEAR), dtype="float32")
    X = np.stack([cargar(c) for c in caps])
    res = {}
    for nombre, m in [("partida", base), ("nuevo", modelo_despliegue)]:
        p = np.mean([m.predict(v, verbose=0) for v in [X, X[:, :, ::-1], X[:, ::-1], X[:, ::-1, ::-1]]], axis=0)
        res[nombre] = p
        print(f"{nombre:8s} capturas={len(caps)} | marcadas malignas (P≥0,495): {100 * ((1 - p[:, 0]) >= 0.495).mean():.1f}% "
              f"| P(maligno) mediana {np.median(1 - p[:, 0]):.2f}")
        if nombre == "nuevo":
            print(f"         con el umbral calibrado de este modelo ({UMBRAL:.3f}): {100 * ((1 - p[:, 0]) >= UMBRAL).mean():.1f}% malignas")
    pd.DataFrame({"archivo": [c.name for c in caps], "pmal_partida": 1 - res["partida"][:, 0],
                  "pmal_nuevo": 1 - res["nuevo"][:, 0]}).to_csv(RUN_DIR / "capturas_capilografo.csv", index=False)
else:
    print("ℹ️ No encontré capturas del capilógrafo en /kaggle/input (opcional).")
del base; gc.collect()


# 10c. TEST (solo desde Z.14)
Evalúa **una vez** el modelo entrenado en esta sesión sobre las 2 146 imágenes de Test, con el sesgo, el umbral y la TTA congelados en Validation. Se reporta como evaluación adicional (el Test ya se había usado para certificar el ensamble Z.2 + Z.3).

In [ ]:
# ============================================================
# 11. TEST — SOLO el modelo entrenado en esta sesión (desde Z.14)
#     Pegar DESPUÉS de la celda de evaluación en Validation (la que calcula SESGO y UMBRAL).
#     Usa el sesgo y el umbral congelados en Validation; el test no se usa para decidir nada.
# ============================================================
import io, re, tarfile, json
from concurrent.futures import ThreadPoolExecutor
from sklearn.metrics import (f1_score, balanced_accuracy_score, roc_auc_score,
                             confusion_matrix, precision_recall_fscore_support)

HACER_TEST = float(re.sub(r"[^0-9.]", "", EXPERIMENTO) or 0) >= 14
if not HACER_TEST:
    print(f"⏭  {EXPERIMENTO}: el test solo se ejecuta desde Z.14.")
else:
    # ---------- 1) filas de test ----------
    if "tablas" in globals() and tablas:                            # Kaggle
        cand = [d for _, d in tablas if "test" in set(d["split"])]
        por_n = {Path(n.split(":")[-1]).name: d for n, d in tablas}
        test_df = por_n["test.csv"] if "test.csv" in por_n else max(cand, key=len)
        test_df = test_df[test_df["split"] == "test"].reset_index(drop=True)
    else:                                                           # Colab
        test_df = pd.read_csv(WORK_ROOT / "00_DATASET" / "test.csv")
    test_df["image_relpath"] = test_df["image_relpath"].astype(str).str.replace("\\", "/", regex=False)
    test_df["image_relpath"] = [r[r.find("images/"):] if "images/" in r else r for r in test_df["image_relpath"]]
    Y_TEST = test_df["indice_clase_4"].astype(int).values
    for col in [c for c in ["image_relpath", "sha256", "group_id_split"] if c in test_df and c in train_df]:
        fuga = set(test_df[col].astype(str)) & (set(train_df[col].astype(str)) | set(val_df[col].astype(str)))
        assert not fuga, f"❌ Fuga con Train/Validation en {col}: {len(fuga)}"
    print(f"Test: {len(test_df):,} imágenes |", dict(zip(CLASES, np.bincount(Y_TEST, minlength=4))))

    # ---------- 2) caché de test (misma adecuación que Train/Validation) ----------
    TEST_CACHE = LOCAL / f"cache_test_{IMG_SIZE}"
    pend = {r for r in test_df["image_relpath"] if not (TEST_CACHE / r).is_file()}
    def guardar_t(rel, datos):
        dst = TEST_CACHE / rel; dst.parent.mkdir(parents=True, exist_ok=True)
        adecuar_imagen(datos).save(dst, "JPEG", quality=95)
    if pend:
        with ThreadPoolExecutor(max_workers=max(4, os.cpu_count() * 2)) as pool:
            if TAR_PATH is None:
                list(pool.map(lambda r: guardar_t(r, (IMAGES_ROOT / r).read_bytes()), sorted(pend)))
            else:
                futs = []
                with tarfile.open(TAR_PATH, "r:") as tar:
                    for m in tar:
                        if not m.isfile(): continue
                        n = m.name.replace("\\", "/"); i = n.find("images/")
                        rel = n[i:] if i >= 0 else None
                        if rel in pend:
                            futs.append(pool.submit(guardar_t, rel, tar.extractfile(m).read()))
                for f in futs: f.result()
    faltan_t = [r for r in test_df["image_relpath"] if not (TEST_CACHE / r).is_file()]
    assert not faltan_t, f"❌ Faltan {len(faltan_t)} imágenes de test"
    rutas_t = [str(TEST_CACHE / r) for r in test_df["image_relpath"]]
    test_ds = tf.data.Dataset.from_tensor_slices(rutas_t).map(leer, num_parallel_calls=AUTOTUNE).batch(64).prefetch(AUTOTUNE)

    # ---------- 3) predicción + métricas con la regla congelada ----------
    def metricas_test(p, y):
        pred = (np.log(p + 1e-9) + SESGO).argmax(1)
        pm = 1 - p[:, 0]; yb = (y > 0).astype(int); pb = (pm >= UMBRAL).astype(int)
        P_, R_, F_, _ = precision_recall_fscore_support(y, pred, labels=[0, 1, 2, 3], zero_division=0)
        return {"macro_f1": f1_score(y, pred, average="macro"), "balanced_acc": balanced_accuracy_score(y, pred),
                "accuracy": float((pred == y).mean()),
                "sens": float(((pb == 1) & (yb == 1)).sum() / yb.sum()),
                "espec": float(((pb == 0) & (yb == 0)).sum() / (1 - yb).sum()),
                "f1_bin": f1_score(yb, pb), "auc_bin": roc_auc_score(yb, pm),
                "nevo_sospechoso": int(((pred == 0) & (pb == 1)).sum()),
                "por_clase": {c: {"P": float(P_[i]), "R": float(R_[i]), "F1": float(F_[i])} for i, c in enumerate(CLASES)},
                "matriz": confusion_matrix(y, pred, labels=[0, 1, 2, 3]).tolist()}

    P_TEST = predecir(modelo_despliegue, test_ds, tta=USA_TTA)
    R_TEST = {"1x": metricas_test(P_TEST, Y_TEST)}
    for z in EVAL_ZOOMS:                                              # robustez de escala en test
        R_TEST[f"{z:g}x"] = metricas_test(predecir(modelo_despliegue, test_ds.map(recorte_central(z)), tta=USA_TTA), Y_TEST)

    print(f"\n{'TEST '+EXPERIMENTO:10s} {'MacroF1':>8s} {'Sens':>6s} {'Espec':>6s} {'AUC':>6s}  F1: " + " ".join(f"{c:>6s}" for c in CLASES))
    for k, m in R_TEST.items():
        print(f"{k:10s} {m['macro_f1']:8.4f} {m['sens']:6.3f} {m['espec']:6.3f} {m['auc_bin']:6.3f}      "
              + " ".join(f"{m['por_clase'][c]['F1']:6.3f}" for c in CLASES))
    print(f"(sesgo {np.round(SESGO, 2)} | umbral {UMBRAL:.3f} | TTA {USA_TTA} — congelados en Validation)")
    print("Referencia Z.2+Z.3 en test: Macro-F1 0.8164 | sens 0.926 | espec 0.925 | AUC 0.973")

    cm = np.array(R_TEST["1x"]["matriz"])
    plt.figure(figsize=(5.5, 4.5)); plt.imshow(cm, cmap="Blues")
    for i in range(4):
        for j in range(4):
            plt.text(j, i, cm[i, j], ha="center", va="center", color="white" if cm[i, j] > cm.max() / 2 else "black")
    plt.xticks(range(4), CLASES); plt.yticks(range(4), CLASES); plt.xlabel("Predicción"); plt.ylabel("Real")
    plt.title(f"{EXPERIMENTO} — Test — Macro-F1 {R_TEST['1x']['macro_f1']:.4f}")
    plt.tight_layout(); plt.savefig(RUN_DIR / "TEST_matriz_confusion.png", dpi=110); plt.show()

    np.savez_compressed(RUN_DIR / "TEST_probabilidades.npz", p=P_TEST, y=Y_TEST,
                        rutas=np.array(test_df["image_relpath"].values))
    (RUN_DIR / "TEST_resultados.json").write_text(json.dumps(
        {"experimento": EXPERIMENTO, "n": int(len(Y_TEST)), "sesgo": SESGO.tolist(), "umbral": UMBRAL,
         "tta": bool(USA_TTA), "resultados": R_TEST}, indent=2, ensure_ascii=False, default=float), encoding="utf-8")
    print("💾", RUN_DIR / "TEST_resultados.json")


# 11. GUARDADO Y TRAZABILIDAD
Se guarda en `EXPERIMENTOS_Z/<TAG>/<fecha>/`:
modelo `.keras` (float32, sin cabeza contrastiva), `.tflite` float16 para la Pi, configuración, métricas, predicciones por imagen, historia y figuras.
Además se agrega una fila a `EXPERIMENTOS_Z/REGISTRO_EXPERIMENTOS.csv` (sirve para los anexos).

In [ ]:
# ---- Modelo de despliegue en float32 (CPU de la Raspberry) ----
keras.mixed_precision.set_global_policy("float32")
_, desp32, _ = construir_modelos()
desp32.set_weights(modelo_despliegue.get_weights())
RUTA_KERAS = RUN_DIR / "dermatec_clasificador.keras"
desp32.save(RUTA_KERAS)

RUTA_TFLITE, tflite_mb, tflite_diff = RUN_DIR / "dermatec_clasificador_fp16.tflite", None, None
try:
    conv = tf.lite.TFLiteConverter.from_keras_model(desp32)
    conv.optimizations = [tf.lite.Optimize.DEFAULT]
    conv.target_spec.supported_types = [tf.float16]
    import contextlib
    with contextlib.redirect_stdout(io.StringIO()), contextlib.redirect_stderr(io.StringIO()):
        RUTA_TFLITE.write_bytes(conv.convert())
    tflite_mb = RUTA_TFLITE.stat().st_size / 1e6
    it = tf.lite.Interpreter(model_path=str(RUTA_TFLITE)); it.allocate_tensors()
    diffs = []
    for r in val_df["local"].values[:16]:
        x = np.asarray(Image.open(r), dtype="float32")[None]
        it.set_tensor(it.get_input_details()[0]["index"], x); it.invoke()
        diffs.append(np.abs(it.get_tensor(it.get_output_details()[0]["index"]) - desp32.predict(x, verbose=0)).max())
    tflite_diff = float(np.max(diffs))
    print(f"✅ TFLite fp16: {tflite_mb:.1f} MB | diferencia máx. vs Keras: {tflite_diff:.4f}")
except Exception as e:
    print("⚠️ TFLite no exportado (el .keras sí quedó guardado):", repr(e)[:300])

def sha256(p):
    h = hashlib.sha256()
    with open(p, "rb") as f:
        for b in iter(lambda: f.read(1 << 20), b""): h.update(b)
    return h.hexdigest()

CONFIG = {k: v for k, v in globals().items() if k.isupper() and isinstance(v, (int, float, str, bool, list, dict))
          and k not in {"CONFIG", "RESULTADOS"}}
CONFIG.update({"pesos_clase": PESOS_CLASE.tolist(), "tar_path": str(TAR_PATH), "tf": tf.__version__,
               "gpu": GPUS[0].name if GPUS else "cpu", "parametros": int(desp32.count_params())})
limpiar = lambda m: {k: v for k, v in m.items() if k not in ("pred", "pred_bin")}
RESULTADOS = {"tag": TAG, "fecha": datetime.now().isoformat(), "mejor_macro_f1_monitor": monitor.mejor,
              "lecturas": {k: limpiar(v) for k, v in lecturas.items()}, "sesgo": SESGO.tolist(), "umbral": UMBRAL,
              "usa_tta": USA_TTA, "sospechosos_nevo": int(sospechosos.sum()),
              "cumple_macro_f1": FINAL["macro_f1"] >= META_MACRO_F1, "cumple_sens": FINAL["sens"] >= META_SENS,
              "keras": str(RUTA_KERAS), "keras_sha256": sha256(RUTA_KERAS), "tflite_mb": tflite_mb,
              "tflite_diff": tflite_diff, "minutos_sesion": (time.time() - T0_SESION) / 60, "test_usado": False}
(RUN_DIR / "config.json").write_text(json.dumps(CONFIG, indent=2, ensure_ascii=False, default=str))
(RUN_DIR / "resultados.json").write_text(json.dumps(RESULTADOS, indent=2, ensure_ascii=False, default=float))

pred_df = pd.DataFrame(P_FINAL, columns=[f"p_{c}" for c in CLASES])
pred_df.insert(0, "image_id", val_df["image_id"].values); pred_df.insert(1, "real", Y_VAL)
pred_df["pred"], pred_df["p_maligno"], pred_df["sospechoso"] = FINAL["pred"], 1 - P_FINAL[:, 0], sospechosos
pred_df.to_csv(RUN_DIR / "predicciones_validation.csv", index=False)

registro = EXP_Z_ROOT / "REGISTRO_EXPERIMENTOS.csv"
fila = {"tag": TAG, "run": RUN_DIR.name, "macro_f1": FINAL["macro_f1"], "sens": FINAL["sens"], "espec": FINAL["espec"],
        **{f"f1_{c}": FINAL["por_clase"][c]["F1"] for c in CLASES}, "epocas": len(monitor.historia),
        "min": RESULTADOS["minutos_sesion"], "tflite_mb": tflite_mb}
pd.DataFrame([fila]).to_csv(registro, mode="a", header=not registro.exists(), index=False)
print("✅ Todo guardado en:", RUN_DIR)

# 10d. REPORTE FINAL (Train / Validation / Test) + PAQUETE PARA LA APP
Se ejecuta después del guardado del modelo (necesita el `.tflite`).

In [ ]:
# ============================================================
# 10d. REPORTE FINAL PARA LA TESIS — Train / Validation / Test
# Curvas de loss y accuracy, matrices de confusión (4 clases y binaria), F1, sensibilidad y especificidad.
# Misma regla en los tres conjuntos y la MISMA de la app: TTA, sesgo y umbral de Validation, salida jerárquica.
# ============================================================
from sklearn.metrics import (f1_score, balanced_accuracy_score, roc_auc_score, confusion_matrix,
                             precision_recall_fscore_support, accuracy_score)
def evaluar(p, y):
    """Regla de la app (jerárquica): P(maligno) ≥ umbral → MALIGNA y clase = la más probable entre Mel/BCC/SCC
    (log p + sesgo); si no → BENIGNA, Nevo. El binario y la clase nunca se contradicen."""
    lp = np.log(p + 1e-9) + SESGO
    pm = 1 - p[:, 0]; yb = (y > 0).astype(int); pb = (pm >= UMBRAL).astype(int)
    pred = np.where(pb == 1, 1 + lp[:, 1:].argmax(1), 0)
    pred_argmax = lp.argmax(1)
    P_, R_, F_, _ = precision_recall_fscore_support(y, pred, labels=[0, 1, 2, 3], zero_division=0)
    return {"n": int(len(y)), "accuracy": accuracy_score(y, pred), "macro_f1": f1_score(y, pred, average="macro"),
            "macro_f1_argmax_sin_jerarquia": f1_score(y, pred_argmax, average="macro"),
            "balanced_acc": balanced_accuracy_score(y, pred),
            "bin_accuracy": accuracy_score(yb, pb), "bin_f1": f1_score(yb, pb),
            "bin_sens": float(((pb == 1) & (yb == 1)).sum() / yb.sum()), "bin_espec": float(((pb == 0) & (yb == 0)).sum() / (1 - yb).sum()),
            "bin_auc": roc_auc_score(yb, pm),
            **{f"F1_{c}": F_[i] for i, c in enumerate(CLASES)}, **{f"Sens_{c}": R_[i] for i, c in enumerate(CLASES)},
            "cm": confusion_matrix(y, pred, labels=[0, 1, 2, 3]).tolist(), "cm_bin": confusion_matrix(yb, pb, labels=[0, 1]).tolist()}

# Train sin aumentación (la misma adecuación que Validation)
train_eval_ds = tf.data.Dataset.from_tensor_slices(train_df["local"].values).map(leer, num_parallel_calls=AUTOTUNE).batch(64).prefetch(AUTOTUNE)
print("Prediciendo Train sin aumentación (unos minutos)...")
P_TRAIN = predecir(modelo_despliegue, train_eval_ds, tta=USA_TTA)
CONJ = {"Train": (P_TRAIN, Y_TRAIN), "Validation": (P_FINAL, Y_VAL)}
if "P_TEST" in globals(): CONJ["Test"] = (P_TEST, Y_TEST)
REP = {k: evaluar(p, y) for k, (p, y) in CONJ.items()}

tabla = pd.DataFrame({k: {m: v for m, v in r.items() if not m.startswith("cm")} for k, r in REP.items()}).T
print(f"\n{EXPERIMENTO} | sesgo {np.round(SESGO, 2)} | umbral P(maligno) {UMBRAL:.3f} | TTA {USA_TTA}")
print(tabla.round(4).to_string())
tabla.to_csv(RUN_DIR / "METRICAS_FINALES_train_val_test.csv")
(RUN_DIR / "METRICAS_FINALES_train_val_test.json").write_text(json.dumps(
    {"experimento": EXPERIMENTO, "sesgo": SESGO.tolist(), "umbral": UMBRAL, "tta": bool(USA_TTA), "partida": CONTINUAR_DESDE,
     "resultados": REP}, indent=2, ensure_ascii=False, default=float), encoding="utf-8")
np.savez_compressed(RUN_DIR / "probabilidades_train_val_test.npz", train=P_TRAIN, y_train=Y_TRAIN, val=P_FINAL, y_val=Y_VAL,
                    **({"test": P_TEST, "y_test": Y_TEST} if "P_TEST" in globals() else {}))

# ---- Curvas de entrenamiento ----
h = pd.DataFrame(monitor.historia)
fig, ax = plt.subplots(1, 3, figsize=(17, 4.2))
ax[0].plot(h["epoca"], h["train_loss_cls"], "o-", label="Entrenamiento"); ax[0].plot(h["epoca"], h["val_loss"], "s-", label="Validación")
ax[0].set_title("Pérdida (entropía cruzada)"); ax[0].set_xlabel("Época"); ax[0].legend(); ax[0].grid(alpha=.3)
ax[1].plot(h["epoca"], h["train_acc"], "o-", label="Entrenamiento"); ax[1].plot(h["epoca"], h["val_acc"], "s-", label="Validación")
ax[1].set_title("Exactitud (4 clases)"); ax[1].set_xlabel("Época"); ax[1].legend(); ax[1].grid(alpha=.3)
ax[2].plot(h["epoca"], h["val_macro_f1"], "s-", color="C2", label="Macro-F1 Val (criterio 1×/2×)")
ax[2].plot(h["epoca"], h["val_sens@0.5"], "^--", color="C3", label="Sensibilidad maligna Val @0,5")
ax[2].set_title("Validación"); ax[2].set_xlabel("Época"); ax[2].legend(); ax[2].grid(alpha=.3)
plt.suptitle(f"{EXPERIMENTO} — curvas de entrenamiento (train con aumentación)"); plt.tight_layout()
plt.savefig(RUN_DIR / "CURVAS_loss_accuracy.png", dpi=150); plt.show()

# ---- Matrices de confusión ----
n = len(REP)
fig, ax = plt.subplots(2, n, figsize=(5.2 * n, 9.5), squeeze=False)
for j, (k, r) in enumerate(REP.items()):
    for fila_, (M, etiquetas, tit) in enumerate([(np.array(r["cm"]), CLASES, f"{k} — 4 clases\nMacro-F1 {r['macro_f1']:.3f} | Acc {r['accuracy']:.3f}"),
                                                 (np.array(r["cm_bin"]), ["Benigno", "Maligno"], f"{k} — binaria\nSens {r['bin_sens']:.3f} | Espec {r['bin_espec']:.3f}")]):
        a = ax[fila_, j]; Mn = M / M.sum(1, keepdims=True)
        a.imshow(Mn, cmap="Blues", vmin=0, vmax=1)
        for i_ in range(len(M)):
            for j_ in range(len(M)):
                a.text(j_, i_, f"{M[i_, j_]}\n{100 * Mn[i_, j_]:.0f}%", ha="center", va="center", fontsize=9,
                       color="white" if Mn[i_, j_] > 0.5 else "black")
        a.set_xticks(range(len(M)), etiquetas, rotation=20); a.set_yticks(range(len(M)), etiquetas)
        a.set_xlabel("Predicción"); a.set_ylabel("Real"); a.set_title(tit, fontsize=10)
plt.suptitle(f"{EXPERIMENTO} — matrices de confusión (umbral {UMBRAL:.3f}, sesgo de Validation)"); plt.tight_layout()
plt.savefig(RUN_DIR / "MATRICES_confusion_train_val_test.png", dpi=150); plt.show()

# ---- Paquete para la App V6 ----
H_V = -(P_FINAL * np.log2(P_FINAL + 1e-12)).sum(1); H95 = float(np.percentile(H_V, 95))
PAQ = RUN_DIR / "PAQUETE_APP"; PAQ.mkdir(exist_ok=True)
nombre_tfl = f"{EXPERIMENTO}_{BACKBONE}_{IMG_SIZE}px_fp16.tflite"
if RUTA_TFLITE.is_file(): shutil.copy(RUTA_TFLITE, PAQ / nombre_tfl)
cfg_app = {"nombre": f"DERMATEC_{EXPERIMENTO.replace('.', '')}", "modelos": [{"id": EXPERIMENTO, "archivo": nombre_tfl, "img_size": IMG_SIZE}],
           "clases": ["Nevo", "Melanoma", "Carcinoma basocelular", "Carcinoma escamocelular"], "tta": bool(USA_TTA),
           "sesgo": [float(v) for v in SESGO], "umbral_p_maligno": float(UMBRAL), "entropia_p95_bits": H95,
           "calidad": {"modo": "advertencia", "entropia_max_bits": H95, "area_lesion_max": 0.60, "bordes_tocados_max": 1},
           "metricas": {k: {m: float(v) for m, v in r.items() if not m.startswith("cm")} for k, r in REP.items()}}
(PAQ / "config_dermatec.json").write_text(json.dumps(cfg_app, indent=2, ensure_ascii=False), encoding="utf-8")
print(f"📦 Paquete para la app en {PAQ} (p95 entropía {H95:.2f} bits)")


# 12. RESUMEN Y COMPARACIÓN CON LO ANTERIOR
Mismo Validation (2 156 imágenes) en todos los casos.

In [ ]:
referencias = pd.DataFrame([
    ["J2.1-R (multimodal, máscara+radiomics)", 0.7626, 0.9689, 0.6256],
    ["FT72-POLISH ensemble (mejor histórico)",  0.7766, 0.9604, 0.6597],
    ["X.3 RGB limpio (fine-tuning parcial)",    0.6849, 0.9029, 0.4240],
    ["Y.1 RGB + pesos (interrumpido, plateau)", 0.6848, 0.9534, np.nan],
    ["Z.1 fine-tuning completo (cortado ép 15/32)", 0.7324, 0.9500, 0.5780],
    ["Z.1c continuación de Z.1 (EffNetV2B0, CE)",   0.8117, 0.9520, np.nan],
    ["Z.2 EffNetV2B0 + SupCon (mejor)",             0.8471, 0.9503, 0.7360],
    [f"{TAG} (este)", FINAL["macro_f1"], FINAL["sens"], FINAL["por_clase"]["SCC"]["F1"]],
], columns=["Modelo", "Macro-F1", "Sensibilidad", "F1 SCC"]).set_index("Modelo")
print(referencias.round(4).to_string())

print("\n" + "=" * 100)
print(f"RESULTADO {TAG}")
print("=" * 100)
print(f"Macro-F1       : {FINAL['macro_f1']:.4f}   meta ≥ {META_MACRO_F1}  {'✅' if RESULTADOS['cumple_macro_f1'] else '❌'}")
print(f"Sensibilidad   : {FINAL['sens']:.4f}   meta ≥ {META_SENS}  {'✅' if RESULTADOS['cumple_sens'] else '❌'}")
print(f"Especificidad  : {FINAL['espec']:.4f} | umbral P(maligno) {UMBRAL:.3f} | TTA: {USA_TTA}")
for c in CLASES:
    m = FINAL["por_clase"][c]; print(f"  {c:9s} P={m['P']:.3f} R={m['R']:.3f} F1={m['F1']:.3f}")
print(f"Parámetros     : {desp32.count_params():,} | TFLite fp16: {tflite_mb if tflite_mb is None else round(tflite_mb, 1)} MB")
print(f"Sesión         : {RESULTADOS['minutos_sesion']:.0f} min | 🔒 Test NO usado")
print("Carpeta        :", RUN_DIR)

# 13. EMPAQUETAR PARA DESCARGAR
Genera `/kaggle/working/<EXPERIMENTO>_resultados.zip` con todo el RUN_DIR. Luego: panel Output → clic derecho → Download, y **Save Version** (si no, /kaggle/working se borra al cerrar la sesión).

In [ ]:
import shutil
zip_ = shutil.make_archive(f"/kaggle/working/{EXPERIMENTO}_resultados", "zip", RUN_DIR)
print("📦 Descárgalo del panel Output:", zip_)
